In [1]:
import sys
sys.path.append("..")
from experiments.environment import *
init_environment()

import gc
import numpy as np, random

Applied font.family: ['Times New Roman', 'Microsoft YaHei', 'DejaVu Sans']
findfont Times -> C:\Windows\Fonts\times.ttf
findfont CN -> C:\Windows\Fonts\msyh.ttc
🚀 环境初始化完成。


In [2]:
import pandas as pd
import numpy as np
from core.data_processor import TabularDataProcessor
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder

In [3]:
# ========== 1. 构造训练数据（无未见类别，有缺失值） ==========
np.random.seed(42)
n_train = 30

df_train = pd.DataFrame({
    'age': np.random.randint(20, 60, n_train),
    'income': np.random.normal(60000, 20000, n_train).round(0),
    'city': np.random.choice(['北京', '上海', '广州', '深圳'], n_train),
    'education': np.random.choice(['本科', '硕士', '博士'], n_train),
})

# 训练集加入缺失值
df_train.loc[2, 'age'] = np.nan
df_train.loc[5, 'income'] = np.nan
df_train.loc[8, 'city'] = np.nan
df_train.loc[12, 'education'] = np.nan

In [4]:
# ========== 2. 构造测试数据（包含训练未见的类别 '成都' 和 '大专'） ==========
n_test = 10
df_test = pd.DataFrame({
    'age': np.random.randint(20, 60, n_test),
    'income': np.random.normal(60000, 20000, n_test).round(0),
    'city': np.random.choice(['北京', '上海', '广州', '深圳', '成都'], n_test),   # '成都' 未见
    'education': np.random.choice(['本科', '硕士', '博士', '大专'], n_test),     # '大专' 未见
})

# 测试集加入一些缺失值
df_test.loc[1, 'age'] = np.nan
df_test.loc[3, 'income'] = np.nan
df_test.loc[5, 'city'] = np.nan
df_test.loc[7, 'education'] = np.nan
df_test.loc[9, 'age'] = np.nan

print("="*60)
print("训练集概览")
print("="*60)
print(f"shape: {df_train.shape}")
print("city 类别:", df_train['city'].unique())
print("education 类别:", df_train['education'].unique())
print("缺失统计:\n", df_train.isnull().sum())

print("\n"+"="*60)
print("测试集概览")
print("="*60)
print(f"shape: {df_test.shape}")
print("city 类别:", df_test['city'].unique())   # 应包含 '成都'
print("education 类别:", df_test['education'].unique())  # 应包含 '大专'
print("缺失统计:\n", df_test.isnull().sum())

训练集概览
shape: (30, 4)
city 类别: ['北京' '广州' nan '深圳' '上海']
education 类别: ['硕士' '本科' '博士' nan]
缺失统计:
 age          1
income       1
city         1
education    1
dtype: int64

测试集概览
shape: (10, 4)
city 类别: ['北京' '广州' '深圳' '成都' nan]
education 类别: ['本科' '博士' '硕士' '大专' nan]
缺失统计:
 age          2
income       1
city         1
education    1
dtype: int64


In [5]:
# ========== 3. 初始化处理器（使用 ordinal 模式，支持未知类别） ==========
num_cols = ['age', 'income']
cat_cols = ['city', 'education']

processor = TabularDataProcessor(
    num_cols=num_cols,
    cat_cols=cat_cols,
    num_transform_method='quantile',   # 对数值做分布变换
    cat_encoding_method='none',     # 未知类别 -> -1 -> NaN -> MICE
    random_state=42
)

In [6]:
# ========== 4. 在训练集上 fit ==========
processor.fit(df_train)
print("\nFit 完成。编码器见过的类别：")
# 安全打印编码器类别（兼容 LabelEncoder 和 OrdinalEncoder）
print("训练集类别（编码器内存储）：")
for col in cat_cols:
    enc = processor.meta[col]['encoder']
    if isinstance(enc, OrdinalEncoder):
        print(f"{col}: {enc.categories_[0]}")
    elif isinstance(enc, LabelEncoder):
        print(f"{col}: {enc.classes_}")
    else:
        print(f"{col}: (无编码器)")


Fit 完成。编码器见过的类别：
训练集类别（编码器内存储）：
city: ['上海' '北京' '广州' '深圳']
education: ['博士' '本科' '硕士']


In [7]:
# ========== 5. 在测试集上调用 transform() ==========
df_transformed = processor.transform(df_test)

print("\n"+"="*60)
print("transform() 输出（模型输入格式）")
print("="*60)
print("数值列已做 QuantileTransformer → 近似正态分布，均值为0，方差约1")
print("类别列已编码为整数 (0,1,2,...)，未知类别经 MICE 插补为某整数")
print("\n前5行预览：")
print(df_transformed.head(10).to_string())
print(f"\n数据类型: {df_transformed.dtypes.unique()}")
print(f"是否有缺失值: {df_transformed.isnull().any().any()}")


transform() 输出（模型输入格式）
数值列已做 QuantileTransformer → 近似正态分布，均值为0，方差约1
类别列已编码为整数 (0,1,2,...)，未知类别经 MICE 插补为某整数

前5行预览：
        age    income  city  education
0  0.897366 -0.611333   1.0        1.0
1 -0.206779 -0.264293   2.0        0.0
2  0.897366  1.333992   1.0        2.0
3 -1.432129  0.022269   3.0        1.0
4  0.801474 -0.296515   2.0        1.0
5  1.175909  0.984595   2.0        0.0
6  0.011696  0.423072   2.0        0.0
7  1.802743  1.387017   2.0        1.0
8 -0.685306 -0.571798   1.0        1.0
9 -0.149053 -0.118770   2.0        1.0

数据类型: [dtype('float32')]
是否有缺失值: False


In [8]:
# ========== 6. 对比 transform_readable() 输出（可读版） ==========
df_readable = processor.transform_readable(df_test)

print("\n"+"="*60)
print("transform_readable() 输出（可读版）")
print("="*60)
print("数值列已恢复原始量纲（年龄、收入），类别列恢复为原始字符串")
print("\n前5行预览：")
print(df_readable.head(10).to_string())
print(f"\n数据类型: {df_readable.dtypes.to_dict()}")


transform_readable() 输出（可读版）
数值列已恢复原始量纲（年龄、收入），类别列恢复为原始字符串

前5行预览：
   age  income city education
0   51   43830   北京        本科
1   41   49965   广州        博士
2   51   78308   北京        硕士
3   23   54511   深圳        本科
4   49   49405   广州        本科
5   56   70265   广州        博士
6   42   61942   广州        博士
7   58   79373   广州        本科
8   34   45959   北京        本科
9   41   53447   广州        本科

数据类型: {'age': dtype('int64'), 'income': dtype('int64'), 'city': dtype('O'), 'education': dtype('O')}


In [9]:
# ========== 7. 重点检查 transform 对未知类别的处理 ==========
print("\n"+"="*60)
print("专门检查 transform 对未知类别的插补结果")
print("="*60)
# 找出测试集中原本是 '成都' 的行
unknown_city_rows = df_test['city'] == '成都'
print(f"测试集中 '成都' 出现的位置及对应原始行：")
print(df_test[unknown_city_rows][['city', 'age', 'income', 'education']])

print("\n经过 transform() 后，这些行的 city 编码值（应为整数）：")
print(df_transformed.loc[unknown_city_rows, 'city'])

print("\n经过 transform_readable() 后，这些行的 city 还原为某个已知类别（因为'成都'无映射）：")
print(df_readable.loc[unknown_city_rows, 'city'])


专门检查 transform 对未知类别的插补结果
测试集中 '成都' 出现的位置及对应原始行：
  city   age   income education
4   成都  49.0  49405.0        大专
9   成都   NaN  53447.0        大专

经过 transform() 后，这些行的 city 编码值（应为整数）：
4    2.0
9    2.0
Name: city, dtype: float32

经过 transform_readable() 后，这些行的 city 还原为某个已知类别（因为'成都'无映射）：
4    广州
9    广州
Name: city, dtype: object


In [10]:
import pandas as pd
import numpy as np
from core.data_processor import TabularDataProcessor

# ========== 1. 构造数据 ==========
np.random.seed(42)
n = 20

df_full = pd.DataFrame({
    'age': np.random.randint(18, 80, n),
    'income': np.random.normal(50000, 15000, n).round(0),
    'city': np.random.choice(['北京', '上海', '广州', '深圳', '成都'], n),
    'education': np.random.choice(['本科', '硕士', '博士', '大专'], n),
})

# 人为添加缺失值
df_full.loc[2, 'age'] = np.nan
df_full.loc[5, 'income'] = np.nan
df_full.loc[8, 'city'] = np.nan
df_full.loc[12, 'education'] = np.nan
df_full.loc[15, 'age'] = np.nan
df_full.loc[17, 'income'] = np.nan

# 制造一个「未见类别」：把第18行的 city 改为 '杭州'
df_full.loc[18, 'city'] = '杭州'   # 这个值在训练集中不存在

# ========== 2. 划分训练集和测试集 ==========
# 训练集：去掉索引 18（'杭州'）和 8（已有缺失 city，但不含未见类别）
train_idx = [i for i in range(n) if i not in [18]]
test_idx = [18]   # 单独测试未知类别的那一行，也可以包含其他正常行

df_train = df_full.loc[train_idx].copy()
df_test  = df_full.loc[test_idx].copy()

print("训练集 city 唯一值：", df_train['city'].unique())
print("测试集 city 值：", df_test['city'].values)

# ========== 3. 初始化处理器（用 ordinal 模式，支持 unknown） ==========
num_cols = ['age', 'income']
cat_cols = ['city', 'education']

processor = TabularDataProcessor(
    num_cols=num_cols,
    cat_cols=cat_cols,
    num_transform_method='quantile',
    cat_encoding_method='ordinal',   # 关键：ordinal 支持 unknown
    random_state=42
)

# ========== 4. 仅在训练集上 fit ==========
processor.fit(df_train)
print("\nFit 完成，编码器见过的类别：")
for col in cat_cols:
    enc = processor.meta[col]['encoder']
    if isinstance(enc, OrdinalEncoder):
        print(f"{col}: {enc.categories_[0]}")
    else:
        print(f"{col}: {enc.classes_}")

# ========== 5. 在测试集上 transform（包含未见类别 '杭州'） ==========
df_test_transformed = processor.transform(df_test)
print("\n测试集 transform 结果（注意 city 列原本是 '杭州'，编码后应为 NaN 然后被插补）：")
print(df_test_transformed)

# 观察编码后的原始值（在 df_temp 阶段 city 应为 NaN）
# 我们可以手动检查中间过程：但这里直接看最终输出，city 不应该报错，且应该是整数

训练集 city 唯一值： ['上海' '深圳' '北京' nan '成都' '广州']
测试集 city 值： ['杭州']

Fit 完成，编码器见过的类别：
city: ['上海' '北京' '广州' '成都' '深圳']
education: ['博士' '大专' '本科' '硕士']

测试集 transform 结果（注意 city 列原本是 '杭州'，编码后应为 NaN 然后被插补）：
        age    income  city  education
18 -0.15425 -1.016187   2.0        1.0


In [11]:
%run tests/test_processor_with_csv.py --csv tests/framingham.csv --method quantile --noise 0.1

数据集: tests/framingham.csv  shape=(4240, 16)
数值列 (8): ['age', 'cigsPerDay', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose']
类别列 (7): ['male', 'education', 'currentSmoker', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes']

── 数值列结构检测结果 ──
  age                  quantize_step=None    precision=0  is_int=True
  cigsPerDay           quantize_step=None    precision=0  is_int=True
  totChol              quantize_step=None    precision=0  is_int=True
  sysBP                quantize_step=0.5     precision=1  is_int=False
  diaBP                quantize_step=0.5     precision=1  is_int=False
  BMI                  quantize_step=None    precision=2  is_int=False
  heartRate            quantize_step=None    precision=0  is_int=True
  glucose              quantize_step=None    precision=0  is_int=True

加噪强度: 10%，注入 NaN 比例: 2%

── 逆变换结构约束验证 ──
  ✓ age                  ─ 连续列（无量化约束）
  ✓ cigsPerDay           ─ 连续列（无量化约束）
  ✓ totChol              ─ 连续列（无量化约束）
  ✓ sysBP               

In [12]:
%run tests/test_quantize_alignment.py

── 检测结构 ──
  sysBP    quantize_step=0.5     precision=1
  age      quantize_step=None    precision=0
  bmi      quantize_step=None    precision=2

── sysBP（q=0.5）量化对齐 ──
  ✓  sysBP 127.3 → 127.5
  ✓  sysBP 120.2 → 120.0
  ✓  sysBP 130.7 → 130.5
  ✓  sysBP 99.8 → 100.0
  ✓  sysBP 140.25 → 140.0
  5/5 通过

── age（整数列）量化对齐 ──
  ✓  age 3.7 → 4
  ✓  age 2.2 → 2
  ✓  age 47.6 → 48
  ✓  age 64.9 → 65
  ✓  age 30.1 → 30
  5/5 通过

── bmi（连续列）不做量化，只做精度截断 ──
  ✓  bmi 25.134 → 25.13
  ✓  bmi 18.401 → 18.4
  ✓  bmi 30.082 → 30.08
  ✓  bmi 27.991 → 27.99
  ✓  bmi 22.713 → 22.71
  5/5 通过

── 验证不依赖训练集中出现过的值 ──
  ✓  130.7 → 130.5（训练集中无此值，确认非查表）
  ✓  99.8 → 100.0（格点对齐，非最近训练样本）
  2/2 通过
